In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

train_df = pd.read_csv('dataset/splits_sample/train.csv')
test_df = pd.read_csv('dataset/splits_sample/test.csv')

train_df['content_clean_stem'] = train_df['content_clean_stem'].fillna('')
test_df['content_clean_stem'] = test_df['content_clean_stem'].fillna('')

label_mapping = {
    'reliable': 0, 'political': 0, 'bias': 0,
    'fake': 1, 'conspiracy': 1, 'rumor': 1, 'clickbait': 1, 
    'junksci': 1, 'unreliable': 1, 'hate': 1
}

train_df = train_df[~train_df['type'].isin(['unknown', 'satire'])].copy()
test_df = test_df[~test_df['type'].isin(['unknown', 'satire'])].copy()

train_df['label'] = train_df['type'].map(label_mapping)
test_df['label'] = test_df['type'].map(label_mapping)

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df['content_clean_stem'])
X_test_tfidf = tfidf.transform(test_df['content_clean_stem'])

y_train = train_df['label']
y_test = test_df['label']

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.92      0.89     31754
           1       0.92      0.85      0.88     30624

    accuracy                           0.89     62378
   macro avg       0.89      0.89      0.89     62378
weighted avg       0.89      0.89      0.89     62378

